This is the Jupyter Notebook used for the paper "CHIME: A Constrained Harmonic Inductive Bias
for Melody Extraction"

This notebook contains all necessary information to reproduce the results reported in the paper, including the data tables and figures.

In [ ]:
# Import everything needed

import os
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"
import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
import numpy as np
import librosa
from nnAudio.Spectrogram import CQT2010v2
import pandas as pd
import mir_eval
from scipy.io.wavfile import write
from IPython.display import Audio

def gpu(i=0):
    return torch.device(f'cuda:{i}' if torch.cuda.is_available() else 'cpu')
gpu()

In [ ]:
# Define all necessary path variables
storeHCQTPath=''
trainXDataPath='/mnt/SSD/processedData/trainXData/' # example path
trainYDataPath='/mnt/SSD/processedData/trainYData/'

mdbMidi='replace with actual path/Melody/Melody2/'
mdbWav = 'replace with actual path/medleyWav/'

mirexMidi='replace with actual path/mirex05TrainFiles/midi/'
wavPath = 'replace with actual path/mirex05TrainFiles/wavFiles/'

adcMidi='replace with actual path/ADC2004/midi/'
adcwavPath = 'replace with actual path/ADC2004/wavFiles/'

In [ ]:
# Load the model
import torch
import torch.nn as nn
import torch.nn.functional as 

class CHIME(nn.Module): 

    def __init__(self,lr):
        super().__init__()
        self.lr=lr
        self.currEpoch=0
       
        self.finalLayer = nn.Sequential(
                                        nn.LazyConv2d(12,5,padding='same'),
                                        nn.LazyBatchNorm2d(),
                                        nn.ReLU(),
            
                                        nn.LazyConv2d(6,(53,3),padding='same'),
                                        nn.LazyBatchNorm2d(),
                                        nn.ReLU(),
            
                                        nn.LazyConv2d(4,(3,7),padding='same'),
                                        nn.LazyBatchNorm2d(),
                                        nn.ReLU(),
            
                                        nn.LazyConv2d(2,1))
                                       
                                            
            

        self.pitch = nn.Sequential(
                                    nn.ReLU(),nn.Dropout(0.15),
                                        nn.LazyLinear(288))

        self.pitchgru = nn.Sequential(
            nn.ReLU(),nn.Dropout(0.1), nn.LazyLinear(288),
            nn.GRU(
            input_size=288,
            hidden_size=288,
            batch_first=True,
            bidirectional=True)) 



        
        self.analyze = nn.Sequential(
                                        nn.LazyConv2d(12,5,padding=(0,2),stride=(2,1)),
                                        nn.LazyBatchNorm2d(),
                                        nn.ReLU(),
            
                                        nn.LazyConv2d(6,(53,3),padding=(0,1),stride=(3,1)),
                                        nn.LazyBatchNorm2d(),
                                        nn.ReLU(),

                                        nn.LazyConv2d(6,(5,1),stride=(2,1)),
                                        nn.LazyBatchNorm2d(),
                                        nn.ReLU(),
        
                                        nn.AdaptiveMaxPool2d((8,86)))

        
        self.final=nn.Sequential(nn.LazyLinear(16),nn.ReLU(),nn.Dropout(0.2),nn.LazyLinear(1))

        self.voicegru = nn.GRU(
            input_size=48,
            hidden_size=36,
            bidirectional=True,
            batch_first=True,num_layers=3,dropout=0.05)
        
            
    def forward(self, x):

        pitchX = self.finalLayer(x)
        
        B = x.shape[0]

        pitchX=pitchX.flatten(1,2)
        pitchX = pitchX.transpose(1, 2)
        lstmX,_=self.plstm(pitchX)
    
        pitchPred=self.pitch(lstmX)
        
        pitch_map = torch.softmax(pitchPred, dim=-1)
        pitch_map = pitch_map.permute(0, 2, 1)
        pitch_map = pitch_map.unsqueeze(1)
        
        combined = torch.cat([pitch_map, x], dim=1)    

        voiceX = self.analyze(combined)
        voiceX=voiceX.flatten(1,2)
        voiceX = voiceX.permute(0,2,1)

        finalVoice,_ = self.vlstm(voiceX)
        
        finalVoice=self.final(finalVoice)

        return [pitchPred,finalVoice]

        


    def loss(self, predicted, targets, averaged=True):
        
        voicePred = predicted[1]
        pitchPred = predicted[0]

        voiceTarget = (targets >= 0).float()
        pitchTarget = targets
        
        pitchLoss = F.cross_entropy(
            pitchPred.reshape(-1, pitchPred.shape[-1]),
            pitchTarget.reshape(-1),
            ignore_index=-1
        )

        finalLoss = pitchLoss

        if (self.currEpoch>2):
            voiceLoss = F.binary_cross_entropy_with_logits(
                voicePred.squeeze(-1),
                voiceTarget,
                pos_weight=torch.tensor(
                        [1.5],
                        dtype=torch.float32,
                        device=targets.device
                    ),
                reduction='mean'
            )
            
            finalLoss = 4*finalLoss + voiceLoss
      
            

        return finalLoss

            

    def trainStep(self,batch):    
        forwardRes = self.forward(*batch[:-1])
        l = self.loss(forwardRes,batch[-1])
        accuracy = self.valAccuracy(forwardRes,batch[-1])
        return l,accuracy  
    
    def computeValAccuracy(self):
        base_fmin = librosa.note_to_hz('C2') 
        MDBsongArr=[]
        MDBvoiceArr=[]
        MDBtimeArr=[]
        MDBtargArr=[]
        MDBtargTimeArr=[]
        for song in MDBValSongs: # This is defined below
            print(song)
        
            hcqt = np.load(f'{storeHCQTPath}{song}.npy') # The HCQTs should be precomputed for efficiency 
            window = 172
            hop = 86
            
            targetData, freq = targetFreq(f'{mdbMidi}{song}_MELODY2.csv')
            MDBtargTimeArr.append(targetData)
            MDBtargArr.append(freq)
            
            num_frames = hcqt.shape[2]
            num_classes = 288
            
            # Accumulators
            pitch_sum_logits = np.zeros((num_frames, num_classes), dtype=np.float32)
            voice_sum_logits = np.zeros(num_frames, dtype=np.float32)
            counts = np.zeros(num_frames, dtype=np.float32)
            
            for i in range(0, num_frames - window + 1, hop):
            
                x = torch.from_numpy(
                    hcqt[:, :, i:i+window]
                ).unsqueeze(0).float().to(gpu())
            
                pitch_logits, voice_logits = model(x)
            
                pitch_logits = pitch_logits.squeeze(0).detach().cpu().numpy()
            
                voice_logits = voice_logits.squeeze(0).squeeze(-1).detach().cpu().numpy()
            
                pitch_sum_logits[i:i+window] += pitch_logits
                voice_sum_logits[i:i+window] += voice_logits
                counts[i:i+window] += 1
            
            last_start = num_frames - window
            
            x = torch.from_numpy(
                hcqt[:, :, last_start:last_start+window]
            ).unsqueeze(0).float().to(gpu())
            
            pitch_logits, voice_logits = model(x)
            
            pitch_logits = pitch_logits.squeeze(0).detach().cpu().numpy()
            voice_logits = voice_logits.squeeze(0).squeeze(-1).detach().cpu().numpy()
            
            pitch_sum_logits[last_start:last_start+window] += pitch_logits
            voice_sum_logits[last_start:last_start+window] += voice_logits
            counts[last_start:last_start+window] += 1
            
            avg_pitch_logits = pitch_sum_logits / counts[:, None]
            avg_voice_logits = voice_sum_logits / counts
            
            predPitch = np.argmax(avg_pitch_logits, axis=1)
            predictVal = base_fmin * (2.0 ** (predPitch / 48))
            
            probs = 1 / (1 + np.exp(-avg_voice_logits))
            MDBvoiceArr.append(probs)
            MDBsongArr.append(predictVal)
           
            
            pitchTime = np.arange(num_frames) * (512 / 44100)
            MDBtimeArr.append(pitchTime)
            
        bestoa=0
        bestcutoff=0
        for cutoff in range(0,1001):
            cutoff=cutoff/1000
            vr=[]
            vfa=[]
            rca=[]
            rpa=[]
            oa=[]
           
            for i in range(len(MDBtimeArr)):
                predictVal=MDBsongArr[i].copy()
                pitchTime=MDBtimeArr[i]
                voici = MDBvoiceArr[i]
                predVoicing = (voici > cutoff).astype(np.int32)
                predictVal[predVoicing == 0] = 0.0
                ref_v, ref_c, est_v, est_c = mir_eval.melody.to_cent_voicing(
                    MDBtargTimeArr[i],
                    MDBtargArr[i],
                    pitchTime,
                    predictVal,
                    est_voicing=predVoicing
                 )
                
                vrval=mir_eval.melody.voicing_recall(ref_v, est_v)
                vfaval=mir_eval.melody.voicing_false_alarm(ref_v, est_v)
                rcaval = mir_eval.melody.raw_chroma_accuracy(ref_v, ref_c, est_v, est_c)
                rpaval=mir_eval.melody.raw_pitch_accuracy(ref_v, ref_c, est_v, est_c)
                oaval=mir_eval.melody.overall_accuracy(ref_v, ref_c, est_v, est_c)
                
                vr.append(vrval)
                vfa.append(vfaval)
                rca.append(rcaval)
                rpa.append(rpaval)
                oa.append(oaval)
   
            candidateOA = sum(oa)/len(oa)
            if (sum(vr)/len(vr) < 0.2):
                break
            if (candidateOA>bestoa):
                bestoa=candidateOA
                bestcutoff=cutoff
        print(f"Found best oa at cutoff {bestcutoff} and oa {bestoa}")
        return bestoa
    

    def config_optimiser(self):
      return torch.optim.Adam(self.parameters(),lr=self.lr) 
  
from torchinfo import summary
model = Flower(lr=0.001).to(gpu()) # Moved to GPU for summary to avoid device mismatch error
summary(model,input_size=(1,6,288,172))

The following cells are for training the model. 

In [ ]:
# Initialize the data loader


class TrainLoader(Dataset):

    def __init__(self, x_files, y_files):

        self.x_files = x_files
        self.y_files = y_files

        self.index_map = []

        for file_id, path in enumerate(x_files):

            x = np.load(path, mmap_mode='r')

            num_samples = x.shape[0]

            for local_idx in range(num_samples):
                self.index_map.append((file_id, local_idx))

    def __len__(self):
        return len(self.index_map)

    def __getitem__(self, idx):

        file_id, local_idx = self.index_map[idx]

        x = np.load(self.x_files[file_id], mmap_mode='r')
        y = np.load(self.y_files[file_id], mmap_mode='r')
        
        x_sample = x[local_idx]
        y_sample = y[local_idx]
        
        x_sample = torch.from_numpy(x_sample.copy()).float()
        y_sample = torch.from_numpy(y_sample.copy()).long()

        return x_sample, y_sample




trainXFiles=[]
trainYFiles=[]



root = Path(trainXDataPath) 

for a in root.rglob('*'):

    trainXFiles.append(trainXDataPath + a.stem + '.npy')
    trainYFiles.append(trainYDataPath + a.stem + '.npy')

trainDataset = TrainLoader(trainXFiles,trainYFiles)

numWorker = 4

SEED = 3257354076 # This is the seed used to obtain the results published in the paper

print(f"Seed: {SEED}")

np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % (2**32)
    np.random.seed(worker_seed)
    random.seed(worker_seed)

g = torch.Generator()
g.manual_seed(SEED)

trainLoader = DataLoader(
    trainDataset,
    batch_size=256,
    shuffle=True,
    num_workers=numWorker,
    worker_init_fn=seed_worker,
    generator=g,
    pin_memory=False,
    persistent_workers=False

)



In [ ]:
# Initialize the Train class

class Train:
  def __init__(self,max_epochs,gpuOn=False):
    self.me = max_epochs
    self.gpuAct = gpuOn
    self.finaloa=0
    self.waiting=0
      
  def fit(self,model,trainData):
    self.model=model
    self.train_dataloader = trainData

    self.optim = model.config_optimiser()

    self.epoch=0

    for self.epoch in range(self.me):
      print(f"\n--- Epoch {self.epoch+1}/{self.me} ---") #
      self.fit_epoch()
      if (self.waiting==7):
          print("Waiting reached 7 epochs, ending training")
          break


  def fit_epoch(self):
    totalLoss=0
    count=0
    model.train() 
    model.currEpoch+=1
    accu=torch.zeros(28, device=gpu()) 
    for batch in self.train_dataloader:
      if (self.gpuAct):
        batch = [item.to(gpu()) for item in batch]
      loss,acc = self.model.trainStep(batch)
      totalLoss+=loss.detach()
      accu+=acc
      count+=1
      self.optim.zero_grad()
      with torch.no_grad():
        loss.backward()
        self.optim.step()

    self.trainArr.append(totalLoss/count)
    print("Loss of", totalLoss/count)
    print("Train Accuracy array")
    print((accu / count).reshape(7, 4))
    loss=0
    accu=torch.zeros(28, device=gpu()) 
    count=0
    model.eval()
    with torch.no_grad():
        calcoa = self.model.computeValAccuracy()
        if (calcoa>self.finaloa):
          self.finaloa=calcoa
          self.waiting=0
          torch.save(model.state_dict(), '/home/tristan/storeModel/BestCHIME.pt')
          print(f"Saved the best model checkpoint, reached OA: {calcoa}")
        else:
            print(f"Best oa {self.finaloa}, achieved {calcoa}")
            self.waiting+=1
        
        

In [ ]:
# Training the model

def init_weights(m):
     if isinstance(m, torch.nn.Linear) or isinstance(m, torch.nn.Conv2d):
        torch.nn.init.kaiming_normal_(m.weight)
        if m.bias is not None:
            torch.nn.init.zeros_(m.bias)


model = CHIME(lr=0.0005).to(gpu()) 

initializeLayers = torch.randn(1,6, 288, 172).to(gpu())
_ = model(initializeLayers)

model.apply(init_weights)

trainer = Train(300,gpuOn=True)
trainer.fit(model, trainLoader,None)


# Reproducing the paper results

To reproduce the numbers reported in our paper, we provide the model weights for the CHIME model used to obtain the reported metrics, saved in the file named "PaperCHIME.pt". The threshold obtained on the validation set was 0.731. You can also verify this by running the code below.

If you have not stored all of the HCQTs beforehand, replace the "hcqt = np.load(f'{storeHCQTPath}{song}.npy')" line with the code in the next cell

In [ ]:
# This option uses the librosa library to generate the HCQTs, which was used for all models in the paper besides "CHIME-Fast"
harmonics = [0.5, 1, 2, 3, 4, 5]
wavSong,sr = librosa.load(f'{mdbWav}{song}.wav',sr=44100)
        
hcqt = []

base_fmin = librosa.note_to_hz('C2')

for h in harmonics:
  cqt_h = librosa.cqt(
      wavSong,
      sr=sr,
      n_bins=288,
      bins_per_octave=48,
      fmin=h * base_fmin,
      hop_length=512
  )

  cqt_h=librosa.amplitude_to_db(np.abs(cqt_h),ref=np.max)
  cqt_h = np.maximum(cqt_h, -80.0)
  cqt_h = (cqt_h + 80) / 80
  hcqt.append(cqt_h)

min_frames = min(h.shape[-1] for h in hcqt)
hcqt = [h[:, :min_frames] for h in hcqt]
hcqt = np.stack(hcqt, axis=0)

In [ ]:
# This option uses the nnAudio library to quickly generate the HCQTs, as used for "CHIME-Fast"
harmonics = [0.5, 1, 2, 3, 4, 5]
wavSong,sr = librosa.load(f'{mdbWav}{song}.wav',sr=44100)
        
x = torch.tensor(wavSong, dtype=torch.float32).unsqueeze(0).to(gpu())

cqts = [
    CQT2010v2(
        sr=44100,
        hop_length=512,
        fmin=h * base_fmin,
        n_bins=288,
        bins_per_octave=48
    ).to(gpu())
    for h in harmonics
]

hcqt = []

for cqt in cqts:
    cqt_h = cqt(x).squeeze(0)

    cqt_h = 20 * torch.log10(
        torch.clamp(cqt_h, min=1e-10)
    )
    cqt_h = cqt_h - cqt_h.max()
    cqt_h = torch.clamp(cqt_h, min=-80.0)
    cqt_h = (cqt_h + 80.0) / 80.0

    hcqt.append(cqt_h)

hcqt = torch.stack(hcqt)

In [ ]:
model = CHIME(lr=0.0005).to(gpu()) 
model.load_state_dict(torch.load("replace this with the path to the model weights"))
bestcutoff = 0.731 '''Avoids errors in case you do not run the below cell. If you 
run the below cell, this value will be overwritten with the actual value obtained'''

In [ ]:
# Verify that the threshold used maximizes performance on the validation set

MDBValSongs = [
    "SecretMountains_HighHorse",
    "ClaraBerryAndWooldog_WaltzForMyVictims",
    "NightPanther_Fire",
    "MusicDelta_Gospel",
    "AvaLuna_Waterduct",
    "CelestialShore_DieForUs",
    "MusicDelta_ChineseHenan",
    "MusicDelta_Vivaldi",
    "MusicDelta_ChineseJiangNan",
    "MusicDelta_FusionJazz",
    "MusicDelta_ChineseDrama"
    ]


MDBsongArr=[]
MDBvoiceArr=[]
MDBtimeArr=[]
MDBtargArr=[]
MDBtargTimeArr=[]

model.eval()

for song in MDBValSongs:
    print(song)

    hcqt = np.load(f'{storeHCQTPath}{song}.npy')
    
    window = 86
    hop = 43
    
    targetData, freq = targetFreq(f'{mdbMidi}{song}_MELODY2.csv')
    MDBtargTimeArr.append(targetData)
    MDBtargArr.append(freq)
    
    num_frames = hcqt.shape[2]
    num_classes = 288
    
    pitch_sum_logits = np.zeros((num_frames, num_classes), dtype=np.float32)
    voice_sum_logits = np.zeros(num_frames, dtype=np.float32)
    counts = np.zeros(num_frames, dtype=np.float32)
    
    for i in range(0, num_frames - window + 1, hop):
    
        x = torch.from_numpy(
            hcqt[:, :, i:i+window]
        ).unsqueeze(0).float().to(gpu())
    
        pitch_logits, voice_logits = model(x)
    
        pitch_logits = pitch_logits.squeeze(0).detach().cpu().numpy()
    
        voice_logits = voice_logits.squeeze(0).squeeze(-1).detach().cpu().numpy()
    
        pitch_sum_logits[i:i+window] += pitch_logits
        voice_sum_logits[i:i+window] += voice_logits
        counts[i:i+window] += 1
    
    last_start = num_frames - window
    
    x = torch.from_numpy(
        hcqt[:, :, last_start:last_start+window]
    ).unsqueeze(0).float().to(gpu())
    
    pitch_logits, voice_logits = model(x)
    
    pitch_logits = pitch_logits.squeeze(0).detach().cpu().numpy()
    voice_logits = voice_logits.squeeze(0).squeeze(-1).detach().cpu().numpy()
    
    pitch_sum_logits[last_start:last_start+window] += pitch_logits
    voice_sum_logits[last_start:last_start+window] += voice_logits
    counts[last_start:last_start+window] += 1
    
    avg_pitch_logits = pitch_sum_logits / counts[:, None]
    avg_voice_logits = voice_sum_logits / counts
    
    predPitch = np.argmax(avg_pitch_logits, axis=1)
    predictVal = base_fmin * (2.0 ** (predPitch / 48))
    
    probs = 1 / (1 + np.exp(-avg_voice_logits))

    MDBvoiceArr.append(probs)
    MDBsongArr.append(predictVal)

    
    pitchTime = np.arange(num_frames) * (512 / 44100)
    MDBtimeArr.append(pitchTime)
    
bestoa=0
bestcutoff=0
for cutoff in range(1001):
    cutoff=cutoff/1000
    vr=[]
    vfa=[]
    rca=[]
    rpa=[]
    oa=[]
   
    for i in range(len(MDBtimeArr)):
        predictVal=MDBsongArr[i].copy()
        pitchTime=MDBtimeArr[i]
        voici = MDBvoiceArr[i]
        predVoicing = (voici > cutoff).astype(np.int32)
        predictVal[predVoicing == 0] = 0.0
        ref_v, ref_c, est_v, est_c = mir_eval.melody.to_cent_voicing(
            MDBtargTimeArr[i],
            MDBtargArr[i],
            pitchTime,
            predictVal,
            est_voicing=predVoicing
         )
        
        vrval=mir_eval.melody.voicing_recall(ref_v, est_v)
        vfaval=mir_eval.melody.voicing_false_alarm(ref_v, est_v)
        rcaval = mir_eval.melody.raw_chroma_accuracy(ref_v, ref_c, est_v, est_c)
        rpaval=mir_eval.melody.raw_pitch_accuracy(ref_v, ref_c, est_v, est_c)
        oaval=mir_eval.melody.overall_accuracy(ref_v, ref_c, est_v, est_c)
        
        vr.append(vrval)
        vfa.append(vfaval)
        rca.append(rcaval)
        rpa.append(rpaval)
        oa.append(oaval)

    candidateOA = sum(oa)/len(oa)

    if (candidateOA>bestoa):
        bestoa=candidateOA
        bestcutoff=cutoff
print(f"Found best oa at cutoff {bestcutoff} and oa {bestoa}")

# Use this "bestcutoff" as the cutoff value for the test datasets

In [ ]:
# Evaluate CHIME or its variants


for song in MDBTestSongs:
            print(song)

            hcqt = np.load(f'{storeHCQTPath}{song}.npy')
            window = 86
            hop = 43
            
            targetData, freq = targetFreq(f'{mdbMidi}{song}_MELODY2.csv')
            MDBtargTimeArr.append(targetData)
            MDBtargArr.append(freq)
            
            num_frames = hcqt.shape[2]
            num_classes = 288
            
            pitch_sum_logits = np.zeros((num_frames, num_classes), dtype=np.float32)
            voice_sum_logits = np.zeros(num_frames, dtype=np.float32)
            counts = np.zeros(num_frames, dtype=np.float32)
            
            for i in range(0, num_frames - window + 1, hop):
            
                x = torch.from_numpy(
                    hcqt[:, :, i:i+window]
                ).unsqueeze(0).float().to(gpu())
            
                pitch_logits, voice_logits = model(x)
            
                pitch_logits = pitch_logits.squeeze(0).detach().cpu().numpy()
            
                voice_logits = voice_logits.squeeze(0).squeeze(-1).detach().cpu().numpy()
            
                pitch_sum_logits[i:i+window] += pitch_logits
                voice_sum_logits[i:i+window] += voice_logits
                counts[i:i+window] += 1
            
            last_start = num_frames - window
            
            x = torch.from_numpy(
                hcqt[:, :, last_start:last_start+window]
            ).unsqueeze(0).float().to(gpu())
            
            pitch_logits, voice_logits = model(x)
            
            pitch_logits = pitch_logits.squeeze(0).detach().cpu().numpy()
            voice_logits = voice_logits.squeeze(0).squeeze(-1).detach().cpu().numpy()
            
            pitch_sum_logits[last_start:last_start+window] += pitch_logits
            voice_sum_logits[last_start:last_start+window] += voice_logits
            counts[last_start:last_start+window] += 1
            
            avg_pitch_logits = pitch_sum_logits / counts[:, None]
            avg_voice_logits = voice_sum_logits / counts
            
            predPitch = np.argmax(avg_pitch_logits, axis=1)
            predictVal = base_fmin * (2.0 ** (predPitch / 48))
            
            probs = 1 / (1 + np.exp(-avg_voice_logits))

            MDBvoiceArr.append(probs)
            MDBsongArr.append(predictVal)
      
            
            pitchTime = np.arange(num_frames) * (512 / 44100)
            MDBtimeArr.append(pitchTime)
    

for song in mirexTrain:
            hcqt = np.load(f'{storeHCQTPath}{song}.npy')
    
            targetData,freq = targetMirex(f'{mirexMidi}{song}REF.txt')
            MIREXtargTimeArr.append(targetData)
            MIREXtargArr.append(freq)
            
            num_frames = hcqt.shape[2]
            num_classes = 288
            
            pitch_sum_logits = np.zeros((num_frames, num_classes), dtype=np.float32)
            voice_sum_logits = np.zeros(num_frames, dtype=np.float32)
            counts = np.zeros(num_frames, dtype=np.float32)
            
            for i in range(0, num_frames - window + 1, hop):
            
                x = torch.from_numpy(
                    hcqt[:, :, i:i+window]
                ).unsqueeze(0).float().to(gpu())
            
                pitch_logits, voice_logits = model(x)
            
                pitch_logits = pitch_logits.squeeze(0).detach().cpu().numpy()
            
                voice_logits = voice_logits.squeeze(0).squeeze(-1).detach().cpu().numpy()
            
                pitch_sum_logits[i:i+window] += pitch_logits
                voice_sum_logits[i:i+window] += voice_logits
                counts[i:i+window] += 1
            
            last_start = num_frames - window
            
            x = torch.from_numpy(
                hcqt[:, :, last_start:last_start+window]
            ).unsqueeze(0).float().to(gpu())
            
            pitch_logits, voice_logits = model(x)
            
            pitch_logits = pitch_logits.squeeze(0).detach().cpu().numpy()
            voice_logits = voice_logits.squeeze(0).squeeze(-1).detach().cpu().numpy()
            
            pitch_sum_logits[last_start:last_start+window] += pitch_logits
            voice_sum_logits[last_start:last_start+window] += voice_logits
            counts[last_start:last_start+window] += 1
            
            avg_pitch_logits = pitch_sum_logits / counts[:, None]
            avg_voice_logits = voice_sum_logits / counts
            
            predPitch = np.argmax(avg_pitch_logits, axis=1)
            predictVal = base_fmin * (2.0 ** (predPitch / 48))
            
            probs = 1 / (1 + np.exp(-avg_voice_logits))

            MIREXvoiceArr.append(probs)
            MIREXsongArr.append(predictVal)

            
            pitchTime = np.arange(num_frames) * (512 / 44100)
            MIREXtimeArr.append(pitchTime)



for song in adcTrain:
                    
            hcqt = np.load(f'{storeHCQTPath}{song}.npy')
    

            targetData,freq = targetADC(f'{adcMidi}{song}REF.txt')
            ADCtargTimeArr.append(targetData)
            ADCtargArr.append(freq)
            
            num_frames = hcqt.shape[2]
            num_classes = 288
            
            pitch_sum_logits = np.zeros((num_frames, num_classes), dtype=np.float32)
            voice_sum_logits = np.zeros(num_frames, dtype=np.float32)
            counts = np.zeros(num_frames, dtype=np.float32)
            
            for i in range(0, num_frames - window + 1, hop):
            
                x = torch.from_numpy(
                    hcqt[:, :, i:i+window]
                ).unsqueeze(0).float().to(gpu())
            
                pitch_logits, voice_logits = model(x)
            
                pitch_logits = pitch_logits.squeeze(0).detach().cpu().numpy()
            
                voice_logits = voice_logits.squeeze(0).squeeze(-1).detach().cpu().numpy()
            
                pitch_sum_logits[i:i+window] += pitch_logits
                voice_sum_logits[i:i+window] += voice_logits
                counts[i:i+window] += 1
            
            last_start = num_frames - window
            
            x = torch.from_numpy(
                hcqt[:, :, last_start:last_start+window]
            ).unsqueeze(0).float().to(gpu())
            
            pitch_logits, voice_logits = model(x)
            
            pitch_logits = pitch_logits.squeeze(0).detach().cpu().numpy()
            voice_logits = voice_logits.squeeze(0).squeeze(-1).detach().cpu().numpy()
            
            pitch_sum_logits[last_start:last_start+window] += pitch_logits
            voice_sum_logits[last_start:last_start+window] += voice_logits
            counts[last_start:last_start+window] += 1
            
            avg_pitch_logits = pitch_sum_logits / counts[:, None]
            avg_voice_logits = voice_sum_logits / counts
            
            predPitch = np.argmax(avg_pitch_logits, axis=1)
            predictVal = base_fmin * (2.0 ** (predPitch / 48))
            
            probs = 1 / (1 + np.exp(-avg_voice_logits))

            ADCvoiceArr.append(probs)
            ADCsongArr.append(predictVal)
      
            
            pitchTime = np.arange(num_frames) * (512 / 44100)
            ADCtimeArr.append(pitchTime)



cutoff=bestcutoff # Assuming you ran the code in the previous cell. If not, replace with 0.731
vr=[]
vfa=[]
rca=[]
rpa=[]
oa=[]
vrT=0
vfaT=0
rcaT=0
rpaT=0
oaT=0

for i in range(len(MDBtimeArr)):
    predictVal=MDBsongArr[i].copy()
    pitchTime=MDBtimeArr[i]
    voici = MDBvoiceArr[i]
    predVoicing = (voici > cutoff).astype(np.int32)
    predictVal[predVoicing == 0] = 0.0
    ref_v, ref_c, est_v, est_c = mir_eval.melody.to_cent_voicing(
        MDBtargTimeArr[i],
        MDBtargArr[i],
        pitchTime,
        predictVal,
        est_voicing=predVoicing
        )
    
    vrval=mir_eval.melody.voicing_recall(ref_v, est_v)
    vfaval=mir_eval.melody.voicing_false_alarm(ref_v, est_v)
    rcaval = mir_eval.melody.raw_chroma_accuracy(ref_v, ref_c, est_v, est_c)
    rpaval=mir_eval.melody.raw_pitch_accuracy(ref_v, ref_c, est_v, est_c)
    oaval=mir_eval.melody.overall_accuracy(ref_v, ref_c, est_v, est_c)
    
    vr.append(vrval)
    vfa.append(vfaval)
    rca.append(rcaval)
    rpa.append(rpaval)
    oa.append(oaval)
vrT+=sum(vr)/len(vr)
vfaT+=sum(vfa)/len(vfa)
rcaT+=sum(rca)/len(rca)
rpaT+=sum(rpa)/len(rpa)
oaT+=sum(oa)/len(oa)

print("---- FINAL MDB ----", cutoff)     
print(f"VR is {sum(vr)/len(vr)}")
print(f"VFA is {sum(vfa)/len(vfa)}")
print(f"RPA is {sum(rpa)/len(rpa)}")
print(f"RCA is {sum(rca)/len(rca)}")
print(f"OA is {sum(oa)/len(oa)}")

vr=[]
vfa=[]
rca=[]
rpa=[]
oa=[]

for i in range(len(MIREXtimeArr)):
    predictVal=MIREXsongArr[i].copy()
    pitchTime=MIREXtimeArr[i]
    voici = MIREXvoiceArr[i]
    predVoicing = (voici > cutoff).astype(np.int32)
    predictVal[predVoicing == 0] = 0.0
    ref_v, ref_c, est_v, est_c = mir_eval.melody.to_cent_voicing(
        MIREXtargTimeArr[i],
        MIREXtargArr[i],
        pitchTime,
        predictVal,
        est_voicing=predVoicing
        )
    
    vrval=mir_eval.melody.voicing_recall(ref_v, est_v)
    vfaval=mir_eval.melody.voicing_false_alarm(ref_v, est_v)
    rcaval = mir_eval.melody.raw_chroma_accuracy(ref_v, ref_c, est_v, est_c)
    rpaval=mir_eval.melody.raw_pitch_accuracy(ref_v, ref_c, est_v, est_c)
    oaval=mir_eval.melody.overall_accuracy(ref_v, ref_c, est_v, est_c)
    
    vr.append(vrval)
    vfa.append(vfaval)
    rca.append(rcaval)
    rpa.append(rpaval)
    oa.append(oaval)
vrT+=sum(vr)/len(vr)
vfaT+=sum(vfa)/len(vfa)
rcaT+=sum(rca)/len(rca)
rpaT+=sum(rpa)/len(rpa)
oaT+=sum(oa)/len(oa)

print("---- FINAL MIREX ----", cutoff)     
print(f"VR is {sum(vr)/len(vr)}")
print(f"VFA is {sum(vfa)/len(vfa)}")
print(f"RPA is {sum(rpa)/len(rpa)}")
print(f"RCA is {sum(rca)/len(rca)}")
print(f"OA is {sum(oa)/len(oa)}")

vr=[]
vfa=[]
rca=[]
rpa=[]
oa=[]

for i in range(len(ADCtimeArr)):
    predictVal=ADCsongArr[i].copy()
    pitchTime=ADCtimeArr[i]
    voici = ADCvoiceArr[i]
    predVoicing = (voici > cutoff).astype(np.int32)
    predictVal[predVoicing == 0] = 0.0
    ref_v, ref_c, est_v, est_c = mir_eval.melody.to_cent_voicing(
        ADCtargTimeArr[i],
        ADCtargArr[i],
        pitchTime,
        predictVal,
        est_voicing=predVoicing
        )
    
    vrval=mir_eval.melody.voicing_recall(ref_v, est_v)
    vfaval=mir_eval.melody.voicing_false_alarm(ref_v, est_v)
    rcaval = mir_eval.melody.raw_chroma_accuracy(ref_v, ref_c, est_v, est_c)
    rpaval=mir_eval.melody.raw_pitch_accuracy(ref_v, ref_c, est_v, est_c)
    oaval=mir_eval.melody.overall_accuracy(ref_v, ref_c, est_v, est_c)
    
    vr.append(vrval)
    vfa.append(vfaval)
    rca.append(rcaval)
    rpa.append(rpaval)
    oa.append(oaval)
vrT+=sum(vr)/len(vr)
vfaT+=sum(vfa)/len(vfa)
rcaT+=sum(rca)/len(rca)
rpaT+=sum(rpa)/len(rpa)
oaT+=sum(oa)/len(oa)

print("---- FINAL ADC ----", cutoff)     
print(f"VR is {sum(vr)/len(vr)}")
print(f"VFA is {sum(vfa)/len(vfa)}")
print(f"RPA is {sum(rpa)/len(rpa)}")
print(f"RCA is {sum(rca)/len(rca)}")
print(f"OA is {sum(oa)/len(oa)}")

print("---- FINAL ----", cutoff)     
print(f"VR is {vrT/3}")
print(f"VFA is {vfaT/3}")
print(f"RPA is {rpaT/3}")
print(f"RCA is {rcaT/3}")
print(f"OA is {oaT/3}")


# Using CHIME to predict the melody of your favorite songs (with the synthesized audio)

For using CHIME to generate a melodic contour, we provide two different model weights in the Github repository.

The first, as mentioned above, is "PaperCHIME.pt". The second, "UltimateCHIME.pt", is a result of training
CHIME on all training, validation, and test datasets. Do not use this version to reproduce the results in the paper.
Instead, we created this version to provide a more accurate model for melody extraction. While "PaperCHIME.pt" already 
performs well (as indicated by the results in the accompanying paper), this version may provide better performance. We recommend
a voicing threshold of 0.3 for "UltimateCHIME.pt"

In [ ]:
model = CHIME(lr=0.0005).to(gpu()) 
model.load_state_dict(torch.load("replace this with the path to the model weights"))

In [ ]:
model.eval()

cutoff = 0.2
base_fmin = librosa.note_to_hz('C2')
window = 172
hop = 86
harmonics = [0.5, 1, 2, 3, 4, 5]



wavSong,sr = librosa.load('Replace this with the file path for your song',sr=44100)
hcqt = []

for h in harmonics:
  cqt_h = librosa.cqt(
      wavSong,
      sr=sr,
      n_bins=288,
      bins_per_octave=48,
      fmin=h * base_fmin,
      hop_length=512
  )

  cqt_h=librosa.amplitude_to_db(np.abs(cqt_h),ref=np.max)
  cqt_h = np.maximum(cqt_h, -80.0)
  cqt_h = (cqt_h + 80) / 80
  hcqt.append(cqt_h)

min_frames = min(h.shape[-1] for h in hcqt)
hcqt = [h[:, :min_frames] for h in hcqt]
hcqt = np.stack(hcqt, axis=0)

num_frames = hcqt.shape[2]
num_classes = 288

pitch_sum_logits = np.zeros((num_frames, num_classes), dtype=np.float32)
voice_sum_logits = np.zeros(num_frames, dtype=np.float32)
counts = np.zeros(num_frames, dtype=np.float32)

for i in range(0, num_frames - window + 1, hop):

    x = torch.from_numpy(
        hcqt[:, :, i:i+window]
    ).unsqueeze(0).float().to(gpu())

    pitch_logits, voice_logits = model(x)

    pitch_logits = pitch_logits.squeeze(0).detach().cpu().numpy()

    voice_logits = voice_logits.squeeze(0).squeeze(-1).detach().cpu().numpy()

    pitch_sum_logits[i:i+window] += pitch_logits
    voice_sum_logits[i:i+window] += voice_logits
    counts[i:i+window] += 1

last_start = num_frames - window

x = torch.from_numpy(
    hcqt[:, :, last_start:last_start+window]
).unsqueeze(0).float().to(gpu())

pitch_logits, voice_logits = model(x)

pitch_logits = pitch_logits.squeeze(0).detach().cpu().numpy()
voice_logits = voice_logits.squeeze(0).squeeze(-1).detach().cpu().numpy()

pitch_sum_logits[last_start:last_start+window] += pitch_logits
voice_sum_logits[last_start:last_start+window] += voice_logits
counts[last_start:last_start+window] += 1

avg_pitch_logits = pitch_sum_logits / counts[:, None]
avg_voice_logits = voice_sum_logits / counts

predPitch = np.argmax(avg_pitch_logits, axis=1)
predictVal = base_fmin * (2.0 ** (predPitch / 48))

probs = 1 / (1 + np.exp(-avg_voice_logits))
predVoicing = (probs > cutoff).astype(np.int32)

predictVal[predVoicing == 0] = np.nan

pitchTime = np.arange(num_frames) * (512 / 44100)




# Synthesizes the predicted melody

f0 = np.nan_to_num(predictVal, nan=0.0).astype(float)
voiced_frames = f0 > 100.0
f0[~voiced_frames] = 0.0

duration_sec = pitchTime[-1]
sample_times = np.arange(int(duration_sec * sr)) / sr

voiced_interp = np.interp(
    sample_times, pitchTime, voiced_frames.astype(float)
)
voiced_mask = voiced_interp >= 0.5

if np.any(voiced_frames):
    phase_f0 = np.interp(
        sample_times, pitchTime[voiced_frames], f0[voiced_frames]
    )
else:
    phase_f0 = np.zeros_like(sample_times)

raw_audio = np.zeros_like(sample_times)
amplitude_env = np.zeros_like(sample_times)
fade_samples = int(sr * 0.010)

voicing_edges = np.diff(
    np.pad(voiced_mask.astype(np.int8), (1, 1), constant_values=0)
)
voiced_starts = np.flatnonzero(voicing_edges == 1)
voiced_ends = np.flatnonzero(voicing_edges == -1)

for start, end in zip(voiced_starts, voiced_ends):
    segment_f0 = phase_f0[start:end]
    segment_phase = np.zeros_like(segment_f0)
    if len(segment_f0) > 1:
        segment_phase[1:] = (
            2 * np.pi * np.cumsum(segment_f0[:-1]) / sr
        )

    segment_env = np.ones_like(segment_f0)
    segment_fade = min(fade_samples, len(segment_f0) // 2)
    if segment_fade > 1:
        fade = 0.5 - 0.5 * np.cos(
            np.linspace(0.0, np.pi, segment_fade)
        )
        segment_env[:segment_fade] *= fade
        segment_env[-segment_fade:] *= fade[::-1]
    elif segment_fade == 1:
        segment_env[0] = 0.0
        segment_env[-1] = 0.0

    raw_audio[start:end] = np.sin(segment_phase)
    amplitude_env[start:end] = segment_env

audio = raw_audio * amplitude_env

max_val = np.max(np.abs(audio))
if max_val > 0:
    audio *= 0.98 / max_val

write("FinalSong.wav", sr, audio.astype(np.float32))
Audio("FinalSong.wav")


# Creating the figure showcasing the predicted melody contours
The below cell is the exact code used to create the figure displaying CHIME's predictions for the song "Lontano" by Matthew
Entwistle from the MedleyDB dataset

In [ ]:
higherBound=52.8
lowerBound=48
predictVal= # Replace with the predicted frequencies for the Lontano song
pitchTime= # Replace with the predicted time steps for the Lontano song
voici = # Replace with the predicted voicing for the Lontano song
predVoicing = (voici > 0.731).astype(np.int32)
predictVal[predVoicing == 0] = np.nan

plt.figure(figsize=(10, 7))
predictVal[predictVal<1]=np.nan

plt.plot(pitchTime, predictVal, linewidth=2, label="Prediction",color='black')

plt.xlabel("Time (s)")
plt.xlim(lowerBound,higherBound)
plt.ylabel("Frequency (Hz)")
plt.ylim(0,1250)
plt.title(song)
plt.legend()
plt.grid(False) 


plt.show()